<div>
<center><img src="../assets/Flux-logo.svg" width="360"/></center>
</div>

<div style="background:linear-gradient(90deg,#036291 0%,#91C2D8 100%);padding:20px 26px;border-radius:10px;border-left:10px solid #D9A441;margin-top:18px">
<h1 style="margin:0;color:#ffffff">Module 2: Fluence in Kubernetes</h1>
<p style="margin:6px 0 0 0;color:#DCECF4;font-size:15px">We will use Usernetes, rootless Kubernetes in Docker on this instance, to install fluence and schedule pods.</p>
<p style="margin:2px 0 0 0;color:#DCECF4;font-size:13px">flux-framework &middot; fluence &middot; usernetes</p>
</div>


# Welcome

[Fluence](https://github.com/converged-computing/fluence) brings Fluxion, the Flux graph scheduler, into Kubernetes as a scheduler plugin. We run it in a [Usernetes](https://github.com/rootless-containers/usernetes) cluster, one Kubernetes node in a rootless docker container on this instance.

Click on <button data-commandLinker-command="terminal:open" data-name="fluence" href="#">this button</button> to open a Terminal and drag it alongside this notebook. Run the commands there.

```bash
cd ~/tutorial/module2
```


## 1. Bring up the cluster

`~/start-usernetes.sh` builds the node image, starts the node, runs `kubeadm init`, installs flannel, and writes `~/.kube/config`. It takes a few minutes.

```bash
bash ~/start-usernetes.sh
```

fluence needs Kubernetes 1.36 with the gang scheduling API on, so the usernetes checkout is pinned to its last 1.36 commit with the `GangScheduling` and `GenericWorkload` gates set. `podgroups` should be in this list.

```bash
kubectl get nodes
kubectl api-resources --api-group=scheduling.k8s.io
```


## 2. Install fluence

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Description:</span> fluence is a scheduler plugin. At startup it builds a Fluxion resource graph from the cluster's nodes, and every pod with <code>schedulerName: fluence</code> is placed by matching a Fluxion jobspec against that graph.
</div>

```bash
cd ~/tutorial/module2
kubectl apply -f manifests/fluence.yaml
kubectl apply -f manifests/fluence-resources.yaml
kubectl rollout status -n kube-system deployment/fluence --timeout=300s
kubectl rollout status -n kube-system deployment/fluence-webhook --timeout=300s
kubectl rollout status -n kube-system daemonset/fluence-deviceplugin --timeout=300s
```

The first manifest is the scheduler, its webhook, and RBAC. The second describes two quantum backends and the device plugin that advertises them. Section 4 uses them.

The scheduler logs the graph it built. The node is there with a vertex per core, and the two backends next to it.

```bash
kubectl logs -n kube-system deployment/fluence | grep -A 30 "RESOURCE GRAPH"
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Tip:</span> The fluence log has the graph, every jobspec it sent, and every allocation it got back. When something does not schedule, look there first.
</div>

## 3. Schedule a gang

A gang is a set of pods that start together or not at all. Kubernetes 1.36 has a native API for it, the `PodGroup`, and fluence places the group against its graph as one jobspec. You do not write the PodGroup. [`manifests/gang.yaml`](manifests/gang.yaml) is an indexed Job of two workers with one label, `fluence.flux-framework.org/group`, and fluence's webhook creates the PodGroup, sizes it from the Job, and links the pods to it. Each worker estimates pi from its own random samples and prints as it goes.

```bash
kubectl apply -f manifests/gang.yaml
kubectl get podgroups.scheduling.k8s.io
kubectl get pods -l job-name=training -o wide
kubectl logs -l job-name=training -f
```

The PodGroup the webhook made has `minCount: 2`, and both pods landed on the node in one match. List it while the workers run. When every member has finished, fluence deletes the PodGroups it created and frees the allocation with them. The jobspec is in the fluence log, two slots of one core each.

```bash
kubectl logs -n kube-system deployment/fluence | grep -A 30 "MATCH 1/1: submitting" | head -40
```

Now the same gang asking for five hundred cores per pod, [`manifests/gang-too-big.yaml`](manifests/gang-too-big.yaml).

```bash
kubectl apply -f manifests/gang-too-big.yaml
kubectl get podgroups.scheduling.k8s.io
kubectl logs -n kube-system deployment/fluence | grep "MATCH 1/1 FAILED" | tail -2
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Notice:</span> Nothing is bound, not one pod of the two. Fluxion matches the gang as a unit, so there is never half a gang holding the node while the other half waits. The same idea as Module 1's held classical job.
</div>

Deleting a PodGroup or a pod frees its allocation. A PodGroup can stick in `Terminating` on its finalizer, and the second command clears it.

```bash
kubectl delete -f manifests/gang-too-big.yaml
kubectl patch podgroup toobig --type=merge -p '{"metadata":{"finalizers":null}}' 2>/dev/null || true
```

I (@vsoch) needed the second command frequently during development!

## 4. The hybrid pair

This is Module 1's pair in Kubernetes, [`manifests/hybrid-gang.yaml`](manifests/hybrid-gang.yaml), an indexed Job of two with `coordination: shared`. Index 0 is the scout. It acquires the session and runs its circuit, and a sidecar next to it watches the task. Index 1 is the worker. The webhook puts a scheduling gate on it, so it is not even scheduled until the scout's task reaches the device. Then the sidecar removes the gate and hands it the scout's session id, and it submits its own circuit into that session. The scout keeps the session open until the worker is done, then releases it. The scout holds one core while it waits in the vendor queue and the worker holds nothing until the device is ready, which is the same design as Module 1 with the eventlog memo replaced by a pod annotation.

The devices come from the ConfigMap you applied in section 2. It describes `ibm_fez` and `ibm_marrakesh` as `qdevice` vertices with `vendor` and `qrmi_type` attributes. The scout asks for any `qpu`, fluence picks one, and the webhook hands the name to both pods as `FLUXION_BACKEND`.

### Without credentials

First as a rehearsal, like the mock in Module 1. [`manifests/hybrid-rehearsal.yaml`](manifests/hybrid-rehearsal.yaml) is the same pair with one more label, `fluence.flux-framework.org/rehearsal: "true"`. The scout prints the backend it was given and exits without submitting. The sidecar releases the worker with no session, and the worker does the same. Nothing touches IBM.

The label is what makes this safe. Without it, a scout that exits without submitting is treated as a failure, for example missing credentials. The worker stays gated and the sidecar exits with an error, so nothing runs by accident.

```bash
kubectl apply -f manifests/hybrid-rehearsal.yaml
kubectl get pods -l app=hybrid -w
```

The worker shows `SchedulingGated` until the scout exits, then runs and completes. Two PodGroups come out of one Job, `hybrid-scout` for the scout alone and `hybrid` for the workers. Read the three logs.

```bash
kubectl get podgroups.scheduling.k8s.io
kubectl logs -l app=hybrid,batch.kubernetes.io/job-completion-index=0 -c sampler
kubectl logs -l app=hybrid,batch.kubernetes.io/job-completion-index=0 -c fluence-sidecar
kubectl logs -l app=hybrid,batch.kubernetes.io/job-completion-index=1 -c sampler
```

```bash
kubectl delete -f manifests/hybrid-rehearsal.yaml
```

### With credentials

The manifest without the label, with a Secret, opens a session and runs the circuits, which costs QPU seconds. You need an IBM Cloud api key and the CRN of your Quantum instance. This is a new terminal, so export them here even if you did in Module 1. The backends in the ConfigMap have to be ones your plan can open a session on. Edit it if not, and restart the scheduler and webhook with `kubectl rollout restart -n kube-system deployment/fluence deployment/fluence-webhook`.

```bash
export IBM_CLOUD_TOKEN=...
ibmcloud login --apikey "$IBM_CLOUD_TOKEN"
export IBM_CLOUD_CRN=$(ibmcloud resource service-instances --service-name quantum-computing --output json | jq -r '.[0].crn')
kubectl create secret generic ibm-quantum --from-literal=token="$IBM_CLOUD_TOKEN" --from-literal=crn="$IBM_CLOUD_CRN"
```

```bash
kubectl apply -f manifests/hybrid-gang.yaml
kubectl get pods -l app=hybrid -w
```

This time the worker stays gated while the scout's task sits in IBM's queue, and is released the moment the task runs. The same three logs show it. The scout acquires and runs. The sidecar sees the task, waits for it, and releases the worker with the session id. The worker has no `acquired` line, only the session it was handed and its own task inside it.

```bash
kubectl logs -l app=hybrid,batch.kubernetes.io/job-completion-index=0 -c sampler
kubectl logs -l app=hybrid,batch.kubernetes.io/job-completion-index=0 -c fluence-sidecar
kubectl logs -l app=hybrid,batch.kubernetes.io/job-completion-index=1 -c sampler
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Notice:</span> In Module 1 you chose the device and exported its variables, and the graph only enforced a quota. Here the devices are modeled, the scheduler chooses, and the workload learns what it got from <code>FLUXION_BACKEND</code>. The credential stays with the workload either way, and in both modules the classical side does not run until the device is ready.
</div>


## 5. Clean up

Remove the workloads first, then the cluster. The last command stops the node container and removes its volumes, so the next `start-usernetes.sh` starts clean.

```bash
kubectl delete -f manifests/gang.yaml -f manifests/hybrid-rehearsal.yaml -f manifests/hybrid-gang.yaml
make -C ~/usernetes down-v
```

<div style="background:#DCECF4;border-left:6px solid #D9A441;padding:12px 18px;color:#06293D"><strong>Module 2 complete</strong></div>
